In [ ]:
# ─── CELL 1: Setup ───────────────────────────────────────────────────────────
from openai import OpenAI
from google.colab import userdata
import pandas as pd
import json
import time

client = OpenAI(api_key=userdata.get('OPENAI_API'))
df = pd.read_excel("claims_for_api.xlsx")

In [ ]:
# ─── CELL 2: Crea il JSONL (formato OpenAI) ──────────────────────────────────
input_jsonl = "batch_paraphrase_input.jsonl"

SYSTEM_PROMPT = (
    "You are a precise and conservative text editor."
    "Your task is to paraphrase  claims while strictly preserving their original meaning, factual content, and verifiability conditions."
)

USER_TEMPLATE = """Rewrite the following claim in different words.

Guidelines:
- Preserve all numbers, dates, statistics, and factual assertions exactly.
- Preserve the original logical direction and strength of the claim (do not reverse, weaken, or hedge it).
- Maintain all information necessary to verify the claim. Do not remove or generalize named entities if this would affect interpretability.
- Keep approximately the same level of specificity and length as the original claim.
- Use a substantially different sentence structure from the original (e.g., change active to passive voice, reorder subject/predicate, or alter the syntactic construction), while keeping the same underlying proposition.
- If the claim is very short, ensure at least three lexical substitutions are made in addition to structural variation.

Before writing, internally ensure that (not include this check in your output):
- the paraphrase expresses the same testable claim,
- no information relevant for fact-checking has been lost or altered

Output:
- A single rewritten claim.
- No preamble, no explanation.

Claim: {claim}"""

with open(input_jsonl, "w") as f:
    for idx, row in df.iterrows():
        claim_text = str(row.get('statement', ""))

        task = {
            "custom_id": f"row_{idx}",          # equivalente di "key" in Gemini
            "method": "POST",
            "url": "/v1/chat/completions",       # endpoint fisso per batch
            "body": {
                "model": "gpt-4o-mini",          # economico, ottimo per paraphrasing
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": USER_TEMPLATE.format(claim=claim_text)}
                ],
                "temperature": 0.6,              # un po' più basso per paraphrasing conservativo
                "max_tokens": 150
            }
        }
        f.write(json.dumps(task) + "\n")

print(f"✅ Creato {input_jsonl} con {len(df)} richieste.")

✅ Creato batch_paraphrase_input.jsonl con 3191 richieste.


In [ ]:
# ─── CELL 3: Upload ──────────────────────────────────────────────────────────
print("Uploading file...")
with open(input_jsonl, "rb") as f:
    uploaded_file = client.files.create(
        file=f,
        purpose="batch"           # OpenAI vuole purpose="batch", non mime_type
    )

print(f"✅ Upload OK.")
print(f"   File ID: {uploaded_file.id}")   # es. file-abc123

Uploading file...
✅ Upload OK.
   File ID: file-2GHuZpL174iYzCU78FfbRG


In [ ]:
# ─── CELL 4: Crea il batch job ───────────────────────────────────────────────
print("Submitting batch job...")
batch_job = client.batches.create(
    input_file_id=uploaded_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h"
)

print(f"🚀 Job creato!")
print(f"   ID:     {batch_job.id}")        # annotati questo se chiudi Colab
print(f"   State:  {batch_job.status}")

Submitting batch job...
🚀 Job creato!
   ID:     batch_6a043f1a39388190aa7d8bb003a484d1
   State:  validating


In [15]:
# ─── RECOVERY ────────────────────────────────────────────────────────────────
from openai import OpenAI
from google.colab import userdata
import pandas as pd
import json
import time

client = OpenAI(api_key=userdata.get('OPENAI_API'))


batch_job = client.batches.retrieve("batch_6a043f1a39388190aa7d8bb003a484d1") #CHANGEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEE!!!!!!!!!!!!!!!
print(f"Stato: {batch_job.status}")
print(f"Completate: {batch_job.request_counts.completed}/{batch_job.request_counts.total}")

Stato: completed
Completate: 3191/3191


In [16]:
# ─── CELL 5: Polling + Download + Excel ──────────────────────────────────────
import time
import json
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment

TERMINAL_STATES = {"completed", "failed", "cancelled", "expired"}
df = pd.read_excel("claims_for_api.xlsx")

# ── 5a. Polling ───────────────────────────────────────────────────────────────
# Se riapri il notebook, commenta la riga sotto e incolla l'ID manualmente:
# batch_job = client.batches.retrieve("batch_XXXXXXXXXXXXXXXXXXXXXXXX")

print("⏳ In attesa del completamento del batch job...")
while True:
    job = client.batches.retrieve(batch_job.id)
    state = job.status
    completed  = job.request_counts.completed
    total      = job.request_counts.total
    print(f"  [{time.strftime('%H:%M:%S')}] Stato: {state} — {completed}/{total} completate")
    if state in TERMINAL_STATES:
        break
    time.sleep(60)

# ── 5b. Controllo esito ───────────────────────────────────────────────────────
if state != "completed":
    print(f"❌ Job terminato con stato: {state}")
    print(f"   Errori: {job.request_counts.failed} righe fallite")
    raise SystemExit("Interrotto: il job non è andato a buon fine.")

print("✅ Job completato! Scarico i risultati...")

# ── 5c. Download e parsing ────────────────────────────────────────────────────
result_content = client.files.content(job.output_file_id).text

results = []
errors  = []

for line in result_content.splitlines():
    if not line.strip():
        continue
    obj        = json.loads(line)
    custom_id  = obj.get("custom_id", "")
    try:
        paraphrase = (
            obj["response"]["body"]["choices"][0]["message"]["content"].strip()
        )
        status = "success"
    except (KeyError, IndexError) as e:
        paraphrase = str(obj.get("error", e))
        status     = "error"
        errors.append(custom_id)

    results.append({"custom_id": custom_id, "status": status, "paraphrase": paraphrase})

results_df = pd.DataFrame(results)
print(f"   ✅ Successi: {(results_df['status']=='success').sum()}/{len(results_df)}")
if errors:
    print(f"   ⚠️  Errori ({len(errors)} righe): {errors[:10]}")

# ── 5d. Merge con il dataframe originale ──────────────────────────────────────
df_out = df.copy()
df_out["custom_id"] = [f"row_{i}" for i in df.index]
df_out = df_out.merge(results_df[["custom_id", "paraphrase"]], on="custom_id", how="left")
df_out = df_out.drop(columns=["custom_id"])

# ── 5e. Salvataggio Excel ─────────────────────────────────────────────────────
OUTPUT_XLSX = "API_Paraphrases_Full.xlsx"

wb = Workbook()
ws = wb.active
ws.title = "Statements & Paraphrases"

# Header
HEADER_COLOR = "D9E1F2"
ws["A1"] = "original_claim"
ws["B1"] = "paraphrase"
for cell in [ws["A1"], ws["B1"]]:
    cell.font      = Font(bold=True, name="Arial", size=11)
    cell.fill      = PatternFill("solid", start_color=HEADER_COLOR)
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

ws.column_dimensions["A"].width = 65
ws.column_dimensions["B"].width = 65
ws.row_dimensions[1].height = 20

# Dati
for i, row in df_out.iterrows():
    excel_row = i + 2
    ws.cell(row=excel_row, column=1, value=str(row.get("statement",  "") or ""))
    ws.cell(row=excel_row, column=2, value=str(row.get("paraphrase", "") or ""))

    for col in [1, 2]:
        cell = ws.cell(row=excel_row, column=col)
        cell.font      = Font(name="Arial", size=10)
        cell.alignment = Alignment(wrap_text=True, vertical="top")

    if i % 2 == 1:
        for col in [1, 2]:
            ws.cell(row=excel_row, column=col).fill = PatternFill(
                "solid", start_color="F2F2F2"
            )

ws.freeze_panes = "A2"
wb.save(OUTPUT_XLSX)

print(f"\n💾 Salvato: {OUTPUT_XLSX}")
print(f"   Righe totali: {len(df_out)}")
print(f"\nAnteprima:")
print(df_out[["statement", "paraphrase"]].head(3).to_string())

⏳ In attesa del completamento del batch job...
  [10:04:40] Stato: completed — 3191/3191 completate
✅ Job completato! Scarico i risultati...
   ✅ Successi: 3191/3191

💾 Salvato: API_Paraphrases_Full.xlsx
   Righe totali: 3191

Anteprima:
                                                                                                      statement                                                                                                                                             paraphrase
0  Regarding sexual assault against women, Its actually safer not to be in college than it is to be in college.                       When it comes to sexual assault targeting women, being outside of college is statistically safer than being enrolled in college.
1                                            Perdue mismanaged Pillowtex, and nearly 8,000 people got laid off.                                                              Pillowtex was mismanaged by Perdue, resulting in the layoffs of